# Semana 2: Estadística Avanzada y Probabilidad Aplicada (I)

## Objetivos de la Práctica:
1. **Estadística Descriptiva:** Resumir numérica y analíticamente la tendencia central y la dispersión de una variable continua.
2. **Distribución Normal Teórica:** Comparar la distribución empírica de los datos frente a una curva Gaussiana ideal ($N(\mu, \sigma)$).
3. **Distribuciones Discretas:** Simular y comparar distribuciones de **Binomial** y **Poisson** usando `scipy.stats`.
4. **Probabilidad Aplicada (Teorema de Bayes):** Implementar un modelo conceptual a priori y a posteriori para clasificación de correos Spam.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns

# Estilo de gráficos
sns.set_theme(style="whitegrid")

# Carga de datos
df = pd.read_csv("video_games_sales.csv")
col = df.select_dtypes("number").columns[0]  # Selecciona 'Rank'
datos = df[col].dropna()

## 1. Estadística Descriptiva (Medidas de Tendencia Central y Dispersión)

In [ ]:
# Cálculo de métricas básicas
media = datos.mean()
mediana = datos.median()
moda = datos.mode()[0]
desv_est = datos.std()
varianza = datos.var()

print(f"--- Estadística Descriptiva de '{col}' ---")
print(f"Media (Promedio):      {media:.4f}")
print(f"Mediana (Percentil 50): {mediana:.4f}")
print(f"Moda (Valor frecuente): {moda:.4f}")
print(f"Desviación Estándar:   {desv_est:.4f}")
print(f"Varianza:              {varianza:.4f}")

### Interpretación de Resultados Descriptivos:
- La **media** (15.50) y la **mediana** (15.50) son idénticas en esta muestra de ranking continuo del 1 al 30, indicando una distribución simétrica uniforme en los valores de posición.
- La **desviación estándar** (8.8034) refleja la dispersión uniforme del ranking. Al elegir métricas descriptivas, la mediana resulta más robusta ante sesgos o *outliers* cuando analizamos variables con sesgo positivo como `Global_Sales`.

## 2. Histograma y Curva Normal Teórica

In [ ]:
# Visualización de la distribución empírica vs. Curva Gaussiana Teórica
plt.figure(figsize=(9, 5))

# Histograma con densidad de probabilidad (density=True)
plt.hist(datos, bins=15, density=True, alpha=0.6, color="skyblue", edgecolor="black", label="Datos empíricos (Histograma)")

# Curva normal teórica calculada con scipy.stats.norm
x = np.linspace(datos.min(), datos.max(), 200)
pdf_normal = stats.norm.pdf(x, loc=media, scale=desv_est)
plt.plot(x, pdf_normal, 'r-', lw=2.5, label=f"Normal teórica N(μ={media:.1f}, σ={desv_est:.1f})")

plt.title(f"Distribución de {col} vs. Normal Teórica", fontsize=13)
plt.xlabel(col)
plt.ylabel("Densidad de Probabilidad")
plt.legend()
plt.tight_layout()
plt.savefig("distribucion_normal_vs_empirica.png")
plt.show()

### Interpretación del Ajuste a la Normal:
¿La variable se parece a una distribución normal?
- No estrictamente. La variable `Rank` por definición es una secuencia uniforme (1 a N). La curva campana de Gauss teórica asume una mayor concentración de densidad alrededor de la media (centro), mientras que en los datos empíricos la densidad es plana y constante en cada rango de valores.

## 3. Simulación de Distribuciones Discretas (Binomial y Poisson)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 3.1) Distribución Binomial: n=20 ensayos, p=0.35 prob. de éxito
n_trials, p_success = 20, 0.35
x_binom = np.arange(0, n_trials + 1)
pmf_binom = stats.binom.pmf(x_binom, n_trials, p_success)

axes[0].bar(x_binom, pmf_binom, color="#2b5c8f", alpha=0.75, edgecolor="black")
axes[0].set_title(f"Distribución Binomial (n={n_trials}, p={p_success})", fontsize=12)
axes[0].set_xlabel("Número de Éxitos (k)")
axes[0].set_ylabel("Probabilidad P(X = k)")
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# 3.2) Distribución de Poisson: lambda=4.5 eventos por intervalo
mu_poisson = 4.5
x_poisson = np.arange(0, 15)
pmf_poisson = stats.poisson.pmf(x_poisson, mu_poisson)

axes[1].bar(x_poisson, pmf_poisson, color="#d95f02", alpha=0.75, edgecolor="black")
axes[1].set_title(f"Distribución de Poisson (λ={mu_poisson})", fontsize=12)
axes[1].set_xlabel("Número de Eventos por Intervalo (k)")
axes[1].set_ylabel("Probabilidad P(X = k)")
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig("distribuciones_discretas.png")
plt.show()

### Interpretación de Distribuciones Discretas:
1. **Distribución Binomial:** Modela el número de éxitos en $n$ ensayos independientes de Bernoulli. En ciencia de datos se utiliza para experimentos A/B (ej. cuántos usuarios hacen clic en una conversión de 20 impresiones).
2. **Distribución de Poisson:** Modela la frecuencia de ocurrencia de eventos independientes en un intervalo continuo fijo (ej. solicitudes por segundo a un servidor web o número de descargas de un videojuego por hora).

## 4. Teorema de Bayes: Clasificador de Spam Conceptual

In [ ]:
# Definición de probabilidades a priori y condicionales
p_spam = 0.4                  # Probabilidad a priori de que un correo sea Spam: P(Spam)
p_gratis_dado_spam = 0.8       # Verosimilitud: P('gratis' | Spam)
p_gratis_dado_no_spam = 0.1    # Verosimilitud: P('gratis' | No Spam)

p_no_spam = 1.0 - p_spam

# Probabilidad total de la evidencia: P('gratis') mediante Ley de Probabilidad Total
p_gratis = (p_gratis_dado_spam * p_spam) + (p_gratis_dado_no_spam * p_no_spam)

# Aplicación del Teorema de Bayes para obtener la Probabilidad A Posteriori: P(Spam | 'gratis')
p_spam_dado_gratis = (p_gratis_dado_spam * p_spam) / p_gratis

print("=== CLASIFICADOR NAIVE BAYES (CONCEPTUAL) ===")
print(f"Probabilidad a priori P(Spam):           {p_spam:.2f}")
print(f"Probabilidad a priori P(No Spam):        {p_no_spam:.2f}")
print(f"Verosimilitud P('gratis' | Spam):         {p_gratis_dado_spam:.2f}")
print(f"Verosimilitud P('gratis' | No Spam):     {p_gratis_dado_no_spam:.2f}")
print(f"Evidencia total P('gratis'):             {p_gratis:.2f}")
print("-" * 45)
print(f"PROBABILIDAD POSTERIOR P(Spam | 'gratis'): {p_spam_dado_gratis:.4f} ({p_spam_dado_gratis*100:.2f}%)")

### Interpretación de la Probabilidad A Posteriori (Teorema de Bayes):
- **Creencia Inicial (A Priori):** Sin haber leído el contenido del correo, la probabilidad base de que cualquier correo sea spam es del **40%** ($P(\text{Spam}) = 0.40$).
- **Actualización con Nueva Evidencia:** Al escanear la palabra clave *'gratis'*, la verosimilitud favorece fuertemente a la clase Spam. Aplicando el Teorema de Bayes, la probabilidad posterior $P(\text{Spam} \mid \text{'gratis'})$ se eleva drásticamente al **84.21%**.
- **Conclusión:** El Teorema de Bayes permite actualizar cuantitativamente nuestras creencias iniciales a medida que se observa nueva evidencia, siendo el principio matemático fundamental de los algoritmos de filtrado y clasificación inteligente.